# Raster zusammenführen und Punkte erstellen

In [15]:
import rasterio
import rasterio.warp
import rasterio.features
import geopandas as gpd
import numpy as np
import math
import os
import scipy.ndimage as ndi
from rasterio.enums import Resampling
from skimage.morphology import disk  # Für die Puffer-Analyse im Raster
from shapely.geometry import Point, LineString, shape
from tqdm import tqdm
import pickle 



# --- 1. Konfiguration ---


# --- Eingabedateien ---
# 30m ALBA-Raster (Werte 1-15 = Jahre 2010-2024)
ALBA_DISTURBANCE_PATH = r"C:\Users\ge48neq\T-EDGE\Fichtelgebirge\01_Rohdaten\Alba\disturbance_alba_epsg25832_v2.tif"
# 10m Thonfeld-Raster (Werte 1-96 = Monate seit Sep 2017)
THONFELD_DISTURBANCE_PATH = r"C:\Users\ge48neq\T-EDGE\Fichtelgebirge\01_Rohdaten\FrankThonfeld\FCCL_092017-082025_KreiseFichtelgebirge.tif"
# GeoPackage, das die finale Ausdehnung vorgibt
TEMPLATE_GPKG_PATH = r"C:\Users\ge48neq\T-EDGE\Fichtelgebirge\01_Rohdaten\Lancover_bkg_kreis_selected.gpkg"

# Pfad zur Wald-Offenland-Karte (1 = Wald, 0 = Offenland)
WALD_OFFENLAND_PATH = r"C:\Users\ge48neq\T-EDGE\Fichtelgebirge\01_Rohdaten\Holzbodenkarte\holzbodenkarte_2018_32632.tif"

# --- Ausgabe-Pfade ---
OUTPUT_DIR = r"C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2"

# 1. Zwischenprodukt: Kombinierte Störung (Pixel = Jahr)
OUTPUT_DISTURBANCE_YEAR_PATH = os.path.join(OUTPUT_DIR, "disturbance_combined_pixel_year_10m.tif")
# 2. Zwischenprodukt: Finale Landbedeckungskarte (0=Offenland, 1=Wald, 2=Störung)
OUTPUT_LANDCOVER_PATH = os.path.join(OUTPUT_DIR, "final_landcover_map_10m.tif")
# 3. Zwischenprodukt: Finale Landbedeckungskarte (NACH 3-Pixel-Filter)
OUTPUT_LANDCOVER_PATH = os.path.join(OUTPUT_DIR, "final_landcover_map_filtered_10m.tif")
OUTPUT_LANDCOVER_UNFILTERED_PATH = os.path.join(OUTPUT_DIR, "final_landcover_map_unfiltered_10m.tif")

# 4. Endprodukt: Vektorlinien der Transekte
OUTPUT_TRANSECTS_PATH = os.path.join(OUTPUT_DIR, "fichtelforst.gpkg")
# 5. Endprodukt: Vektor-Startpunkte der Transekte
OUTPUT_START_POINTS_PATH = os.path.join(OUTPUT_DIR, "fichtelforst.gpkg")
OUTPUT_START_POINTS_LAYER = "punkte_westpixel"


# --- Logik-Parameter ---
TARGET_CRS = "EPSG:25832"
TARGET_RES_XY = (10.0, 10.0) # 10m Auflösung
NODATA_VALUE = -9999
MIN_DISTURBANCE_SIZE_PX = 3 # Flächen kleiner 3 Pixel werden entfernt

# Kodierung für die finale Landcover-Karte
LC_NONFOREST = 0
LC_FOREST = 1
LC_DISTURBANCE = 2

# Jahre für die Filterung
ALBA_START_YEAR = 2015
ALBA_END_YEAR = 2016
THONFELD_START_YEAR = 2017

# Distanzen für die Filterung (in Pixeln, bei 10m Auflösung)
START_POINT_BUFFER_PX = 10 # 100m
TRANSECT_BUFFER_PX = 20 # 200m
TRANSECT_LENGTH_PX = 10 # 100m

# --- 2. Hilfsfunktionen zur Jahres-Umrechnung ---

def map_alba_to_year(value):
    """Rechnet Alba-Wert (1=2010) in Jahr um."""
    if value >= 1 and value <= 15: # 1-15
        return float(value + 2009)
    return 0.0

def map_thonfeld_to_year(month_value):
    """Rechnet Thonfeld-Monatswert (1-96) in Jahr um."""
    if month_value >= 1 and month_value <= 96:
        year = 2017 + math.floor((month_value - 1 + 8) / 12)
        return float(year)
    return 0.0

# Vektorisiere die Funktionen für Numpy-Arrays
v_map_alba = np.vectorize(map_alba_to_year)
v_map_thonfeld = np.vectorize(map_thonfeld_to_year)

# --- 3. Hilfsfunktion zur Raster-Aufbereitung ---

def reproject_and_match(src_path, bounds, target_crs, target_res, resampling_method, nodata_val, master_profile=None):
    """
    Projiziert ein Quell-Raster auf ein Ziel-Grid.
    Wenn master_profile=None, wird das Grid aus src berechnet (für den 1. Aufruf).
    Wenn master_profile=vorhanden, wird dieses Grid erzwungen.
    """
    
    with rasterio.open(src_path) as src:
        src_crs = src.crs
        src_transform = src.transform
        
        if master_profile:
            # Template vorhanden
            # Erzwinge die Dimensionen und Transformation des Masters.
            dst_transform = master_profile['transform']
            dst_width = master_profile['width']
            dst_height = master_profile['height']
            
            # Erstelle Profil-Kopie für diesen Aufruf
            dst_profile = master_profile.copy()
            dst_profile.update({
                'dtype': np.float32,
                'nodata': nodata_val
            })
        
        else:
            # 1. Aufruf
            # Berechne das Grid basierend auf den Bounds.
            dst_transform, dst_width, dst_height = rasterio.warp.calculate_default_transform(
                src_crs,
                target_crs,
                src.width,
                src.height,
                *src.bounds,
                dst_bounds=bounds,
                resolution=target_res
            )
            
            # Erstelle das Profil für Template-Aufruf
            dst_profile = {
                'driver': 'GTiff',
                'crs': target_crs,
                'transform': dst_transform,
                'width': dst_width,
                'height': dst_height,
                'count': 1,
                'dtype': np.float32,
                'nodata': nodata_val
            }

        # Ziel-Array initialisieren
        dst_array = np.empty((dst_height, dst_width), dtype=np.float32)

        # Reprojektion durchführen
        rasterio.warp.reproject(
            source=rasterio.band(src, 1),
            destination=dst_array,
            src_transform=src_transform,
            src_crs=src_crs,
            dst_transform=dst_transform,
            dst_crs=target_crs,
            resampling=resampling_method,
            dst_nodata=nodata_val
        )
    
    # Gebe das Array und das berechnete oder kopierte Profil zurück
    return dst_array, dst_profile

# --- 5. Hilfsfunktion für Puffer-Kernel ---

def create_transect_buffer_kernel(buffer_px, line_px):
    """
    Erstellt eine boolesche Maske (Kernel) in Form eines 
    Runde-Ecken-Puffers um ein horizontales Liniensegment (line_px) 
    und dessen Startpixel (+1).

    Args:
        buffer_px (int): Puffer-Radius in Pixeln (z.B. 20 für 200m)
        line_px (int): Länge der Transekt-Linie in Pixeln (z.B. 10 für 100m)

    Returns:
        np.ndarray: Boolescher 2D-Kernel.
    """
    
    # Das zu puffernde Segment (Linie + Startpixel) ist line_px + 1 Pixel lang
    # z.B. (c-10) bis (c) = 11 Pixel
    segment_len = line_px + 1 
    
    # Kernel-Dimensionen (Bounding Box des Puffers)
    # y-Achse: buffer | center | buffer -> buffer*2 + 1
    y_dim = buffer_px * 2 + 1
    # x-Achse: buffer | segment | buffer -> buffer*2 + segment_len
    x_dim = buffer_px * 2 + segment_len
    
    # Koordinaten-Zentrum im Kernel
    y_center = buffer_px
    x_segment_start = buffer_px
    x_segment_end = buffer_px + line_px # (Segment-Ende ist inklusiv)
    
    # Erstelle Gitter mit Kernel-Koordinaten
    J, I = np.meshgrid(np.arange(x_dim), np.arange(y_dim))
    
    # Berechne Distanz zur y_center-Linie (Achse)
    dy = np.abs(I - y_center)
    
    # Berechne Distanz zum x-Liniensegment (Achse)
    dx = np.zeros_like(J, dtype=float)
    # Links vom Segment
    dx[J < x_segment_start] = x_segment_start - J[J < x_segment_start]
    # Rechts vom Segment
    dx[J > x_segment_end] = J[J > x_segment_end] - x_segment_end
    # (Über/Unter dem Segment ist dx = 0)
    
    # Berechne Euklidische Distanz zu jedem Punkt des Segments
    distance = np.sqrt(dx**2 + dy**2)
    
    # Maske: True wenn Distanz <= Puffer
    kernel = distance <= float(buffer_px)
    
    print(f"  > Puffer-Kernel erstellt (Größe: {kernel.shape}, Distanz: {buffer_px}px)")
    return kernel


# --- 4. Haupt-Skript ---

def main():
    
    # Sicherstellen, dass das Ausgabe-Verzeichnis existiert
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Prüfen, ob die Waldkarte existiert
    if not os.path.exists(WALD_OFFENLAND_PATH):
        print(f"OBACHT FEHLER: Die Wald-Offenland-Karte wurde nicht gefunden unter: {WALD_OFFENLAND_PATH}")
        print("Bitte ergänze den Pfad in der Konfiguration und starte das Skript erneut.")
        return

    print("--- Workflow gestartet ---")
    # --- STEP 1.1: Vorbereitung und Reprojektion ---
    print("STEP 1.1: Lese Template-Bounds und projiziere Raster...")
    
    template_gdf = gpd.read_file(TEMPLATE_GPKG_PATH)
    template_gdf = template_gdf.to_crs(TARGET_CRS)
    target_bounds = template_gdf.total_bounds

    # 1. Aufruf (ALBA): Erstellt das Master-Grid. 
    # 'profile' speichert jetzt die Master-Geometrie.
    alba_arr, profile = reproject_and_match(
        ALBA_DISTURBANCE_PATH, target_bounds, TARGET_CRS, TARGET_RES_XY, 
        Resampling.nearest, 0 # 0 = NoData/Keine Störung
        # master_profile=None ist hier der Default
    )
    
    # 2. Aufruf (Thonfeld): Nutzt das Master-Grid von Alba
    thonfeld_arr, _ = reproject_and_match(
        THONFELD_DISTURBANCE_PATH, target_bounds, TARGET_CRS, TARGET_RES_XY, 
        Resampling.nearest, 0,
        master_profile=profile
    )

    # 3. Aufruf (Wald): Nutzt ebenfalls das Master-Grid von Alba
    wald_arr, _ = reproject_and_match(
        WALD_OFFENLAND_PATH, target_bounds, TARGET_CRS, TARGET_RES_XY, 
        Resampling.nearest, NODATA_VALUE,
        master_profile=profile
    )
    
    # 'profile' ist das Master-Profil für alle weiteren Schritte
    final_transform = profile['transform']
    final_shape = (profile['height'], profile['width'])
    
    # falls der 1. Aufruf (Alba) 
    # einen anderen Dtype hatte (obwohl die Funktion float32 erzwingt)
    profile['dtype'] = np.float32

    # --- STEP 1.2: Störungsjahre kombinieren ---
    print("STEP 1.2: Kombiniere Störungsjahre (Alba + Thonfeld)...")
    
    alba_year_map = v_map_alba(alba_arr)
    thonfeld_year_map = v_map_thonfeld(thonfeld_arr)

    disturbance_year = np.zeros(final_shape, dtype=np.float32)

    mask_thonfeld = (thonfeld_year_map >= THONFELD_START_YEAR)
    disturbance_year[mask_thonfeld] = thonfeld_year_map[mask_thonfeld]

    mask_alba = (alba_year_map >= ALBA_START_YEAR) & (alba_year_map <= ALBA_END_YEAR)
    disturbance_year[mask_alba] = alba_year_map[mask_alba]

    profile['nodata'] = 0.0
    with rasterio.open(OUTPUT_DISTURBANCE_YEAR_PATH, 'w', **profile) as dst:
        dst.write(disturbance_year.astype(np.float32), 1)

    # --- STEP 1.3: Finale Landbedeckungskarte erstellen ---
    print("STEP 1.3: Erstelle finale Landbedeckungskarte (Mit Wald/Offenland/Störung)...")
    
    lc_map = np.full(final_shape, NODATA_VALUE, dtype=np.int16)
    lc_map[wald_arr != 1] = LC_NONFOREST
    lc_map[wald_arr == 1] = LC_FOREST
    lc_map[disturbance_year > 0] = LC_DISTURBANCE

    # Speichere die UNGEFILTERTE Landnutzungskarte
    print(f"  > Speichere UNGEFILTERTE Landnutzungskarte in: {OUTPUT_LANDCOVER_UNFILTERED_PATH}")
    profile['dtype'] = np.int16
    profile['nodata'] = NODATA_VALUE
    with rasterio.open(OUTPUT_LANDCOVER_UNFILTERED_PATH, 'w', **profile) as dst:
        dst.write(lc_map, 1)

    # --- STEP 2.1: Störflächen filtern. Kleiner 3 Pix ---
    print(f"STEP 2.1: Filtere Störflächen kleiner als {MIN_DISTURBANCE_SIZE_PX} Pixel...")
    
    is_disturbance_mask = (lc_map == LC_DISTURBANCE)
    
    structure = ndi.generate_binary_structure(2, 2)
    labeled_array, num_features = ndi.label(is_disturbance_mask, structure=structure)
    
    print(f"  > {num_features} Störflächen gefunden (vor Filterung).")

    labels, counts = np.unique(labeled_array, return_counts=True)
    
    small_labels = labels[counts < MIN_DISTURBANCE_SIZE_PX]
    small_labels = small_labels[small_labels != 0]

    if len(small_labels) > 0:
        lc_map[np.isin(labeled_array, small_labels)] = LC_FOREST
        print(f"  > {len(small_labels)} kleine Flächen entfernt und zu Wald umklassifiziert.")

    is_disturbance_cleaned = (lc_map == LC_DISTURBANCE)
    # Label-Array für Step 2.4 neu berechnen
    labeled_array_cleaned, num_features_cleaned = ndi.label(is_disturbance_cleaned, structure=structure)

    profile['dtype'] = np.int16
    profile['nodata'] = NODATA_VALUE
    with rasterio.open(OUTPUT_LANDCOVER_PATH, 'w', **profile) as dst:
        dst.write(lc_map, 1)


    # 1. Speichere das Array mit den Patch-IDs als NumPy-Datei
    patch_id_array_path = os.path.join(OUTPUT_DIR, "final_patch_ids.npy")
    np.save(patch_id_array_path, labeled_array_cleaned)
    print(f"  > Patch-ID-Array gespeichert in: {patch_id_array_path}")

    # 2. Speichere die Geotransformation (wichtig für x/y -> r/c)
    # (Das 'transform' Objekt serialisierbar machen)
    transform_path = os.path.join(OUTPUT_DIR, "final_transform.pkl")
    # 'final_transform' ist eine Variable aus Step 1.1
    with open(transform_path, 'wb') as f:
        pickle.dump(final_transform, f)
    print(f"  > Geotransformation gespeichert in: {transform_path}")

    # --- STEP 2.2: Westliche Kantenpixel finden ---
    print("STEP 2.2: Finde westliche Kantenpixel der Störflächen...")
    
    is_disturbance = (lc_map == LC_DISTURBANCE)
    is_not_disturbance = (lc_map != LC_DISTURBANCE)
    
    west_neighbor_not_disturbance = np.roll(is_not_disturbance, shift=1, axis=1)
    west_neighbor_not_disturbance[:, 0] = False

    candidate_edge_pixels = is_disturbance & west_neighbor_not_disturbance

    # --- STEP 2.3: Startpixel filtern (100m Puffer) ---
    print(f"STEP 2.3: Filtere Startpixel (kein Offenland im 100m-Radius)...")
    
    is_non_forest = (lc_map == LC_NONFOREST)
    footprint_100m = disk(START_POINT_BUFFER_PX)
    dilated_non_forest_100m = ndi.binary_dilation(is_non_forest, structure=footprint_100m)

    valid_start_pixels_mask = candidate_edge_pixels & ~dilated_non_forest_100m
    
    start_rows, start_cols = np.where(valid_start_pixels_mask)
    print(f"  > {len(start_rows)} valide Startpixel nach 100m-Filterung gefunden.")

    if len(start_rows) == 0:
        print("Keine validen Startpixel gefunden. Workflow wird beendet.")
        return
    
    # --- VORBEREITUNG FÜR STEP 2.4: Puffer-Kernel erstellen ---
    
    # Hole die Array-Dimensionen für Randprüfungen
    max_row, max_col = lc_map.shape
    
    # Definiere Radien/Längen in Pixeln
    BUFFER_RADIUS_PX = 20 # 200m
    TRANSECT_LEN_PX_LINE = 10 # 100m
    
    print("STEP 2.4 (Pre-Cache): Erstelle 200m (20px) Transekt-Puffer-Kernel...")
    
    # Erstelle den Euklidischen/Runden-Kernel.
    # Er puffert um die 10px-Linie UND das 1px-Startpixel (also 11 Pixel Segment)
    BUFFER_KERNEL = create_transect_buffer_kernel(BUFFER_RADIUS_PX, TRANSECT_LEN_PX_LINE)
    
    # Kernel-Offsets (Mittelpunkt relativ zur (0,0)-Ecke des Kernels)
    # (r, c) (Startpixel) entspricht Kernel-Koordinate (y_center, x_segment_end)
    KERNEL_Y_OFFSET = BUFFER_RADIUS_PX # 20
    # Kernel(0,0) ist bei (c - 10(Linie) - 20(Puffer))
    KERNEL_X_OFFSET_LEFT = TRANSECT_LEN_PX_LINE + BUFFER_RADIUS_PX # 10 + 20 = 30
    
    # --- STEP 2.4: Raster-basierte Transekt-Filterung ---
    print(f"STEP 2.4: Generiere und filtere Transekte (Kernel-basiert)...")

    final_transects_data = []

    # Verwende tqdm für den Fortschritt
    for r, c in tqdm(zip(start_rows, start_cols), total=len(start_rows), desc="Filtere Transekte (Raster)"):
        
        # 1. Hole ID der eigenen Störfläche
        current_disturbance_id = labeled_array_cleaned[r, c]
        
        # 2. PRÜFUNG A (Transekt-Linie): 
        if (c - TRANSECT_LEN_PX_LINE) < 0:
            continue # wenn transekt links das Raster verlässt
            
        transect_labels = labeled_array_cleaned[r, (c - TRANSECT_LEN_PX_LINE) : c]
        if np.any(transect_labels > 0):
            continue

        # 3. PRÜFUNG B (200m Runder-Puffer-Check mit Kernel):
        
        # Definiere den absoluten Bounding Box des Puffer-Kernels
        # Y-Achse
        r_min_abs = r - KERNEL_Y_OFFSET # r - 20
        r_max_abs = r + KERNEL_Y_OFFSET + 1 # r + 21 (für 41 Pixel)
        
        # X-Achse
        # Kernel(0,0) -> c - 30
        c_min_abs = c - KERNEL_X_OFFSET_LEFT # c - 30
        c_max_abs = c - KERNEL_X_OFFSET_LEFT + BUFFER_KERNEL.shape[1] # (c - 30) + 51 -> c + 21
        
        # Finde valide (überlappende) Indizes (Clipping am Rasterrand)
        r_min_valid = max(0, r_min_abs)
        r_max_valid = min(max_row, r_max_abs)
        c_min_valid = max(0, c_min_abs)
        c_max_valid = min(max_col, c_max_abs)
        
        # Wenn der Slice ungültige Dimensionen hat (z.B. 0), überspringen
        if (r_max_valid <= r_min_valid) or (c_max_valid <= c_min_valid):
            continue

        # Berechne die entsprechenden Slices für den Kernel selbst
        k_r_min = r_min_valid - r_min_abs
        k_r_max = k_r_min + (r_max_valid - r_min_valid)
        k_c_min = c_min_valid - c_min_abs
        k_c_max = k_c_min + (c_max_valid - c_min_valid)

        # Hole die Slices aus den Haupt-Arrays und dem Kernel
        try:
            lc_map_window = lc_map[r_min_valid:r_max_valid, c_min_valid:c_max_valid]
            labels_window = labeled_array_cleaned[r_min_valid:r_max_valid, c_min_valid:c_max_valid]
            kernel_window = BUFFER_KERNEL[k_r_min:k_r_max, k_c_min:k_c_max]
        except IndexError:
            # Sicherheitsabfang, falls Slicing fehlschlägt
            continue

        # PRÜFUNG B.1: Ist Offenland (LC_NONFOREST) im Kernel-Puffer?
        # Wende boolesche Maske (kernel_window) auf das Array an
        if np.any(lc_map_window[kernel_window] == LC_NONFOREST):
            continue

        # PRÜFUNG B.2: Ist eine ANDERE Störung im Kernel-Puffer?
        # Wende Maske an, um nur Pixel im Puffer zu erhalten
        labels_in_buffer = labels_window[kernel_window]
        
        # Prüfe diese Pixel auf Störungsflächen als der eigenen
        other_disturbance_mask = (labels_in_buffer > 0) & (labels_in_buffer != current_disturbance_id)
        if np.any(other_disturbance_mask):
            continue

        # --- Wenn alle Prüfungen bestanden ---
        # Vektordaten (Punkte, Linien) erstellen
        start_x, start_y = final_transform * (c + 0.5, r + 0.5)
        # 100m = TRANSECT_LEN_PX_LINE (10)
        end_x, end_y = final_transform * (c - TRANSECT_LEN_PX_LINE + 0.5, r + 0.5)
        
        start_point_geom = Point(start_x, start_y)
        transect_line_geom = LineString([start_point_geom, Point(end_x, end_y)])

        final_transects_data.append({
            'geometry': transect_line_geom,
            'start_point_geom': start_point_geom,
            'disturbance_id': current_disturbance_id,
            'start_row': r,
            'start_col': c
        })

    print(f"  > {len(final_transects_data)} valide Transekte nach Raster-Filterung generiert.")

    # --- STEP 2.5: Ergebnisse speichern ---
    if final_transects_data:
        # Transekte speichern
        transects_gdf = gpd.GeoDataFrame(
            final_transects_data, 
            geometry='geometry', 
            crs=TARGET_CRS
        )
        transects_gdf.drop(columns=['start_point_geom'], inplace=True)
        transects_gdf.to_file(OUTPUT_TRANSECTS_PATH, driver="GPKG")
        print(f"  > Finale Transekte gespeichert in: {OUTPUT_TRANSECTS_PATH}")

        # Startpunkte speichern
        start_points_gdf = gpd.GeoDataFrame(
            final_transects_data, 
            geometry='start_point_geom', 
            crs=TARGET_CRS
        )
        start_points_gdf.drop(columns=['geometry'], inplace=True)
        start_points_gdf.to_file(OUTPUT_START_POINTS_PATH, layer=OUTPUT_START_POINTS_LAYER, driver="GPKG")
        print(f"  > Startpunkte gespeichert in: {OUTPUT_START_POINTS_PATH}, Layer: {OUTPUT_START_POINTS_LAYER}")
    else:
        print("Keine Transekte haben alle Filterkriterien erfüllt.")

    print("--- Workflow abgeschlossen ---")


if __name__ == "__main__":
    main()

--- Workflow gestartet ---
STEP 1.1: Lese Template-Bounds und projiziere Raster...
STEP 1.2: Kombiniere Störungsjahre (Alba + Thonfeld)...


c:\Users\ge48neq\T-EDGE\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:2625: RuntimeWarning: invalid value encountered in map_thonfeld_to_year (vectorized)
  outputs = ufunc(*args, out=...)


STEP 1.3: Erstelle finale Landbedeckungskarte (Mit Wald/Offenland/Störung)...
  > Speichere UNGEFILTERTE Landnutzungskarte in: C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\final_landcover_map_unfiltered_10m.tif
STEP 2.1: Filtere Störflächen kleiner als 3 Pixel...
  > 54754 Störflächen gefunden (vor Filterung).
  > 30127 kleine Flächen entfernt und zu Wald umklassifiziert.
  > Patch-ID-Array gespeichert in: C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\final_patch_ids.npy
  > Geotransformation gespeichert in: C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\final_transform.pkl
STEP 2.2: Finde westliche Kantenpixel der Störflächen...
STEP 2.3: Filtere Startpixel (kein Offenland im 100m-Radius)...
  > 73686 valide Startpixel nach 100m-Filterung gefunden.
STEP 2.4 (Pre-Cache): Erstelle 200m (20px) Transekt-Puffer-Kernel...
  > Puffer-Kernel erstellt (Größe: (41, 51), Distanz: 20px)
STEP 2.4: Generiere und 

Filtere Transekte (Raster): 100%|██████████| 73686/73686 [00:02<00:00, 31477.12it/s]


  > 2331 valide Transekte nach Raster-Filterung generiert.
  > Finale Transekte gespeichert in: C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\fichtelforst.gpkg
  > Startpunkte gespeichert in: C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\fichtelforst.gpkg, Layer: punkte_westpixel
--- Workflow abgeschlossen ---


# Hektar der Störfläche ermitteln

In [16]:
import os
import sys
import geopandas as gpd
import rasterio
import numpy as np
import scipy.ndimage as ndi
from rasterio.features import geometry_mask

# --- 1. Konfiguration ---

# Pfad zum Störungsraster (Pixel 2015-2025 sind Störungen)
RASTER_PATH = r"C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\disturbance_combined_pixel_year_10m.tif"

# --- Pfad zu Punkt-Datei
POINTS_PATH = r"C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\fichtelforst.gpkg"
POINTS_LAYER_NAME = "punkte_westpixel" 

# Name der neuen Spalte für die Fläche in Hektar
AREA_COLUMN_NAME = 'Area_ha'

# Name für die Ausgabedatei (wird im selben Ordner wie die Punkt-Datei gespeichert)
OUTPUT_LAYER_NAME = "punkte_westpixel_ha"


# --- 2. Haupt-Skript ---

def add_patch_area_to_points():
    print("--- Starte Flächenberechnung für Störungspunkte ---")
    
    # --- Schritt 1: Raster laden und Störungsflächen segmentieren ---
    try:
        print(f"Lese Raster: {RASTER_PATH}")
        with rasterio.open(RASTER_PATH) as src:
            data = src.read(1)
            transform = src.transform
            crs = src.crs
            profile = src.profile # Metadaten für spätere Nutzung
            
            # Pixel-Fläche berechnen
            pixel_area_m2 = abs(transform.a * transform.e)
            if pixel_area_m2 == 0:
                print("Obacht !!! FEHLER: Pixel-Fläche ist 0. Transformation ungültig.")
                return False
            pixel_area_ha = pixel_area_m2 / 10000.0
            print(f"  Raster CRS: {crs}, Pixel Fläche: {pixel_area_m2} m² ({pixel_area_ha} ha)")

            # Maske für Störungspixel (1-2026)
            clearing_mask = (data >= 1) & (data <= 2026)
            
            # Störungsflächen segmentieren
            print("Segmentiere Störungsflächen...")
            # `structure` definiert Nachbarschaft (hier 8er-Nachbarschaft)
            structure = ndi.generate_binary_structure(2, 2) 
            labeled_patches, num_features = ndi.label(clearing_mask, structure=structure)
            print(f"  {num_features} zusammenhängende Störungsflächen gefunden.")

            if num_features == 0:
                print("OBACHT FEHLER!?: Keine Störungsflächen im Raster gefunden.")
                
                
    except FileNotFoundError:
        print(f"FEHLER: Rasterdatei nicht gefunden: {RASTER_PATH}")
        return False
    except Exception as e:
        print(f"FEHLER beim Lesen oder Segmentieren des Rasters: {e}")
        return False

    # --- Schritt 2: Fläche pro Segment berechnen ---
    print("Berechne Fläche pro Segment...")
    patch_areas_ha = {} # Dictionary: {patch_id: area_in_ha}
    
    if num_features > 0:
        # Finde alle eindeutigen Patch-IDs (außer 0 = Hintergrund)
        unique_labels = np.unique(labeled_patches)
        valid_labels = unique_labels[unique_labels != 0]
        
        # Zähle Pixel pro Patch-ID
        labels, counts = np.unique(labeled_patches, return_counts=True)
        
        # Erstelle das Dictionary
        for label, count in zip(labels, counts):
            if label != 0: # Ignoriere Wald
                 patch_areas_ha[label] = round(count * pixel_area_ha, 4)
                 
        print(f"  Flächen für {len(patch_areas_ha)} Segmente berechnet.")
        
        # Optional: Min/Max Fläche ausgeben
        if patch_areas_ha:
             min_area = min(patch_areas_ha.values())
             max_area = max(patch_areas_ha.values())
             print(f"  Min. Fläche: {min_area:.4f} ha, Max. Fläche: {max_area:.2f} ha")
        
    # --- Schritt 3: Punkte laden und CRS prüfen/anpassen ---
    try:
        print(f"Lese Punkt-Datei: {POINTS_PATH}")
        points_layer_name = None
        if 'POINTS_LAYER_NAME' in globals():
             points_layer_name = POINTS_LAYER_NAME
             print(f"  Verwende Layer: {points_layer_name}")
             
        points_gdf = gpd.read_file(POINTS_PATH, layer=points_layer_name)
        print(f"  {len(points_gdf)} Punkte gefunden.")
        print(f"  Punkte CRS: {points_gdf.crs}")
        
        # CRS vergleichen und ggf. anpassen
        if points_gdf.crs != crs:
            print(f"WARNUNG: CRS der Punkte ({points_gdf.crs}) unterscheidet sich vom Raster CRS ({crs}). Projiziere Punkte neu...")
            try:
                points_gdf = points_gdf.to_crs(crs)
                print(f"  Punkte erfolgreich nach {crs} reprojiziert.")
            except Exception as e:
                print(f"FEHLER bei der CRS-Transformation der Punkte: {e}")
                print("Stelle sicher, dass beide Layer gültige CRS haben.")
                return False
                
    except FileNotFoundError:
        print(f"FEHLER: Punkt-Datei nicht gefunden: {POINTS_PATH}")
        return False
    except Exception as e:
        print(f"FEHLER beim Lesen der Punkt-Datei: {e}")
        return False

    # --- Schritt 4: Fläche für jeden Punkt extrahieren ---
    print("Extrahiere Flächen-IDs für jeden Punkt...")
    point_patch_ids = []
    
    # Extrahiere Koordinaten als Liste von Tupeln
    coords = [(p.x, p.y) for p in points_gdf.geometry]
    
    # Verwendet rasterio.sample, um die Werte effizient zu extrahieren
    # sample liefert einen Generator -zu Liste konvertieren und den 
    # ersten (und einzigen) wert pro Punkt übernehmen
    try:
        # 
        # sampeln aus dem `labeled_patches`-Raster!
        sampled_values = list(src.sample(coords))
        point_patch_ids = [val[0] for val in sampled_values]
        print(f"  Patch-IDs für {len(point_patch_ids)} Punkte extrahiert.")
        
    except Exception as e:
         print(f"FEHLER beim Sampeln der Rasterwerte an Punktpositionen: {e}")
         # Fallback: Manuelle Iteration (langsamer)
         print("Versuche manuelles Sampeln (langsamer)...")
         point_patch_ids = []
         try:
             for x, y in coords:
                 # Koordinaten in Zeile/Spalte umrechnen
                 row, col = src.index(x, y)
                 # Prüfen, ob Index im Raster liegt
                 if 0 <= row < src.height and 0 <= col < src.width:
                      point_patch_ids.append(labeled_patches[row, col])
                 else:
                      point_patch_ids.append(0) # Punkt außerhalb -> ID 0
             print(f"  Manuelles Sampeln für {len(point_patch_ids)} Punkte abgeschlossen.")
         except Exception as e_inner:
             print(f"FEHLER auch beim manuellen Sampeln: {e_inner}")
             return False

    # --- Schritt 5: Flächen zuweisen und speichern ---
    print(f"Weise Flächen der neuen Spalte '{AREA_COLUMN_NAME}' zu...")
    
    # Verwendet die extrahierten Patch-IDs, um die Flächen aus dem Dictionary zu holen
    # .get(id, 0) gibt 0 zurück, falls die ID nicht im Dict ist (z.B. für ID 0 oder Fehler)
    areas_for_points = [patch_areas_ha.get(patch_id, 0) for patch_id in point_patch_ids]
    
    # Fügt die neue Spalte zum GeoDataFrame hinzu
    points_gdf[AREA_COLUMN_NAME] = areas_for_points
    points_gdf.head()
    
    # Pfad für die Ausgabedatei erstellen
    output_path = POINTS_PATH
    
    try:
        print(f"Speichere Ergebnis nach: {output_path} layer: {OUTPUT_LAYER_NAME}...")
        points_gdf.to_file(output_path, layer=OUTPUT_LAYER_NAME, driver="GPKG")
        print("  Erfolgreich gespeichert.")
        return True
    except Exception as e:
        print(f"FEHLER beim Speichern der Ergebnis-Shape-Datei: {e}")
        return False

# --- 3. Skript ausführen ---
if __name__ == "__main__":
    success = add_patch_area_to_points()


    
    print("\n---------------------------------")
    if success:
        print("Skript erfolgreich beendet.")
    else:
        print("Skript mit Fehlern beendet.")


--- Starte Flächenberechnung für Störungspunkte ---
Lese Raster: C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\disturbance_combined_pixel_year_10m.tif
  Raster CRS: EPSG:25832, Pixel Fläche: 100.0 m² (0.01 ha)
Segmentiere Störungsflächen...
  54754 zusammenhängende Störungsflächen gefunden.
Berechne Fläche pro Segment...
  Flächen für 54754 Segmente berechnet.
  Min. Fläche: 0.0100 ha, Max. Fläche: 284.90 ha
Lese Punkt-Datei: C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\fichtelforst.gpkg
  Verwende Layer: punkte_westpixel
  2331 Punkte gefunden.
  Punkte CRS: EPSG:25832
Extrahiere Flächen-IDs für jeden Punkt...
FEHLER beim Sampeln der Rasterwerte an Punktpositionen: Dataset is closed: C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\disturbance_combined_pixel_year_10m.tif
Versuche manuelles Sampeln (langsamer)...
  Manuelles Sampeln für 2331 Punkte abgeschlossen.
Weise Flächen der neuen Spalte 'Area_ha

# Alter, TWI, Slope, Höhe hinzufügen

Da manche Patches sehr groß sind vielleicht mehrere Spalten. 
- ältestes Pixel auf Fläche
- Alter der bis zu 10 Reihen (100 m) umliegenden Pixel 
- alter des Pixels bei Startpunkt

In [17]:
import geopandas as gpd
import rasterio
import rasterio.warp
from rasterio.enums import Resampling
import numpy as np
import pickle
from scipy.ndimage import median as nd_median
from tqdm import tqdm
import os

print("--- Skript für erweiterte Statistik gestartet ---")

# --- 1. Konfiguration ---

# Verzeichnis-Pfade
BASE_DIR = r"C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2"
WBT_DIR = r"C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\DGM\WBT_Output"
DEM_PATH = r"C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\DGM\DGM_merg_10m.tif"

# Eingabe: Punkte mit area
POINTS_PATH = os.path.join(BASE_DIR, "fichtelforst.gpkg")
POINTS_LAYER = "punkte_westpixel_ha"


# Eingabe: Von Schritt A erstellte Dateien
PATCH_ID_ARRAY_PATH = os.path.join(BASE_DIR, "final_patch_ids.npy")
TRANSFORM_PATH = os.path.join(BASE_DIR, "final_transform.pkl")

# Eingabe: Raster, die gesampelt werden sollen
RASTER_TO_SAMPLE = {
    "Hoehe": DEM_PATH,
    "SCA": os.path.join(WBT_DIR, "02_wbt_sca.tif"),
    "Slope": os.path.join(WBT_DIR, "03_wbt_slope_radians.tif"),
    "TWI": os.path.join(WBT_DIR, "04_wbt_TWI.tif"),
    "Year": os.path.join(BASE_DIR, "disturbance_combined_pixel_year_10m.tif")
}

# Parameter für Nachbarschafts-Median
NEIGHBORHOOD_RADIUS_PX = 10 # 10 Pixel = 100m

# Ausgabe
OUTPUT_PATH = POINTS_PATH
OUTPUT_LAYER = "punkte_westpixel_ha_stats"


# --- 2. Daten laden ---

print(f"Lade Punkte: {POINTS_PATH}")
try:
    points_gdf = gpd.read_file(POINTS_PATH, layer=POINTS_LAYER)
except Exception as e:
    print(f"FEHLER: Konnte Layer '{POINTS_LAYER}' nicht laden. Lade stattdessen 'valid_transect_start_points'...")
    try:
        points_gdf = gpd.read_file(
            os.path.join(BASE_DIR, "valid_transect_start_points.gpkg"), 
            layer="valid_transect_start_points"
        )
    except Exception as e_inner:
        print(f"Konnte Original-Punkte Gpkg nicht laden: {e_inner}")
        exit()

print(f"Lade Patch-ID-Array: {PATCH_ID_ARRAY_PATH}")
patch_id_array = np.load(PATCH_ID_ARRAY_PATH)

print(f"Lade Master-Transformation: {TRANSFORM_PATH}")
with open(TRANSFORM_PATH, 'rb') as f:
    # Dies ist die 'Master'-Transformation
    transform = pickle.load(f)

# --- Master-CRS ermitteln ---
master_crs = None
year_raster_path = RASTER_TO_SAMPLE.get("Year")
if not year_raster_path:
    print("FEHLER: 'Year'-Raster muss in RASTER_TO_SAMPLE definiert sein.")
    exit()

try:
    with rasterio.open(year_raster_path) as src:
        master_crs = src.crs
        print(f"Master-CRS (aus 'Year'-Raster) ermittelt: {master_crs}")
except Exception as e:
    print(f"FEHLER: Konnte 'Year'-Raster nicht öffnen, um Master-CRS zu lesen: {e}")
    exit()

# --- Punkte in Master-CRS reprojizieren ---
if points_gdf.crs != master_crs:
    print(f"Projiziere Punkte von {points_gdf.crs} nach {master_crs}...")
    points_gdf = points_gdf.to_crs(master_crs)
else:
    print("Punkte sind bereits im Master-CRS.")

# Lade alle Raster-Arrays
raster_data = {}
for name, path in RASTER_TO_SAMPLE.items():
    print(f"Lade Raster-Array: {name}...")
    with rasterio.open(path) as src:
        raster_data[name] = {
            "array": src.read(1),
            "nodata": src.nodata
        }

print("Alle Daten geladen.")

# --- 3. Pixel-Koordinaten (r, c) für Punkte berechnen ---
print("Berechne Pixel-Koordinaten (r, c) für alle Punkte...")

# weil points_gdf im Master-CRS ist, sind sie (x, y)
coords_xy_master = [(p.x, p.y) for p in points_gdf.geometry]

# und 'transform' die Master-Transformation ist, sind sie (r, c)
px_coords_master = [rasterio.transform.rowcol(transform, x, y) for x, y in coords_xy_master]

# Speichere die Indizes im GDF
points_gdf['px_row'] = [r for r, c in px_coords_master]
points_gdf['px_col'] = [c for r, c in px_coords_master]

print("Pixel-Koordinaten erfolgreich berechnet.")

# --- 4. Metriken berechnen ---
# Hole die Array-Dimensionen und das Profil des Master-Grids
max_row, max_col = patch_id_array.shape
master_profile = {
    'driver': 'GTiff',
    'crs': master_crs,
    'transform': transform,
    'width': max_col,
    'height': max_row,
    'count': 1,
    'dtype': np.float32, # Ziel-Dtype für Reprojektion
    'nodata': -9999
}
master_shape = (max_row, max_col)

# Hole (x, y) Koordinaten für rasterio.sample
coords = [(p.x, p.y) for p in points_gdf.geometry]

# Iteriere über jedes zu sampelnde Raster
for name, data in raster_data.items():
    print(f"--- Verarbeite Raster: {name} ---")
    
    # Hole die Quelldaten
    source_array = data['array']
    source_nodata = data['nodata']
    
    # --- METRIK 1: Wert am exakten Pixel (via rasterio.sample) ---
    print("Berechne Metrik 1: Punktwert...")
    col_name_point = f"{name}_Point"
    
    # Die Quelldatei erneut öffnen, um .sample() zu verwenden
    with rasterio.open(RASTER_TO_SAMPLE[name]) as src:
        # src.sample() ist robust gegen Grid-Mismatches
        sampled_values = list(src.sample(coords))
        point_values = [val[0] for val in sampled_values]
        
    points_gdf[col_name_point] = point_values

    # --- Vorbereitung für Metrik 2 & 3: Reprojektion ---
    
    # Prüfe, ob das Quell-Raster bereits auf dem Master-Grid liegt
    if source_array.shape == master_shape:
        print("  > Raster-Grid stimmt bereits überein. Überspringe Reprojektion.")
        current_array_aligned = source_array
        current_nodata_aligned = source_nodata
    else:
        print(f"  > Reprojiziere Raster '{name}' auf Master-Grid...")
        
        # Erstelle ein leeres Array mit der Ziel-Shape
        current_array_aligned = np.empty(master_shape, dtype=master_profile['dtype'])
        current_nodata_aligned = master_profile['nodata']

        # Reprojiziere das Quell-Array
        with rasterio.open(RASTER_TO_SAMPLE[name]) as src:
            rasterio.warp.reproject(
                source=rasterio.band(src, 1),
                destination=current_array_aligned,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=master_profile['transform'],
                dst_crs=master_profile['crs'],
                resampling=Resampling.bilinear, # 'nearest' für kategoriale Daten
                dst_nodata=current_nodata_aligned
            )

    # --- METRIK 2: Median der gesamten Störungsfläche (Zonal-Statistik) ---
    print("Berechne Metrik 2: Median der gesamten Patch-Fläche (Zonal)...")
    col_name_patch = f"{name}_Patch_Median"
    
    # Maske für valide Pixel (im reprojizierten Array!)
    valid_mask = (current_array_aligned != current_nodata_aligned) & \
                 (current_array_aligned != source_nodata) & \
                 (patch_id_array > 0)
    
    patch_ids_unique = np.unique(patch_id_array[patch_id_array > 0])
    
    if len(patch_ids_unique) > 0:
        # Berechne Mediane
        median_values = nd_median(
            current_array_aligned[valid_mask], 
            labels=patch_id_array[valid_mask], 
            index=patch_ids_unique
        )
        # Erstelle Lookup-Tabelle
        patch_median_lookup = dict(zip(patch_ids_unique, median_values))
    else:
        patch_median_lookup = {}
    
    # Weise den Median-Wert basierend auf der Patch-ID des Punktes zu
    # (Benötigt werden die (r,c) Koordinaten des MASTER-Grids)
    px_coords_master = [(r, c) for r, c in zip(points_gdf['px_row'], points_gdf['px_col'])]
    point_patch_ids = [patch_id_array[r, c] for r, c in px_coords_master]
    points_gdf[col_name_patch] = [patch_median_lookup.get(pid, np.nan) for pid in point_patch_ids]

    # --- METRIK 3: Median der 100m-NACHBARSCHAFT (nur auf Patch) ---
    print("Berechne Metrik 3: Median der 100m-Nachbarschaft (auf Patch)...")
    col_name_local = f"{name}_Neighborhood_Median"
    
    local_median_values = []
    
    for r, c in tqdm(px_coords_master, desc=f"Lokale Mediane ({name})"):
        try:
            current_patch_id = patch_id_array[r, c]
            if current_patch_id == 0: 
                local_median_values.append(np.nan)
                continue

            # Fenster (10-Pixel-Radius)
            r_min = max(0, r - NEIGHBORHOOD_RADIUS_PX)
            r_max = min(max_row, r + NEIGHBORHOOD_RADIUS_PX + 1)
            c_min = max(0, c - NEIGHBORHOOD_RADIUS_PX)
            c_max = min(max_col, c + NEIGHBORHOOD_RADIUS_PX + 1)
            
            # Slices holen (vom ALIGNED Array!)
            patch_id_window = patch_id_array[r_min:r_max, c_min:c_max]
            value_window = current_array_aligned[r_min:r_max, c_min:c_max]
            
            # Maske: 
            mask = (patch_id_window == current_patch_id) & \
                   (value_window != current_nodata_aligned) & \
                   (value_window != source_nodata)
            
            if np.any(mask):
                local_median_values.append(np.median(value_window[mask]))
            else:
                local_median_values.append(np.nan)
        
        except Exception:
            local_median_values.append(np.nan)
            
    points_gdf[col_name_local] = local_median_values

print("--- Alle Berechnungen abgeschlossen ---")

# --- 5. Speichern ---
print(f"Speichere finale GeoPackage-Datei in: {OUTPUT_PATH}, Layer: {OUTPUT_LAYER}")
# Lösche temporäre Spalten
if 'px_row' in points_gdf.columns:
    points_gdf = points_gdf.drop(columns=['px_row', 'px_col'])

points_gdf.to_file(OUTPUT_PATH, layer=OUTPUT_LAYER, driver="GPKG")
print("--- Skript erfolgreich beendet ---")

--- Skript für erweiterte Statistik gestartet ---
Lade Punkte: C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\fichtelforst.gpkg
Lade Patch-ID-Array: C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\final_patch_ids.npy
Lade Master-Transformation: C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\final_transform.pkl
Master-CRS (aus 'Year'-Raster) ermittelt: EPSG:25832
Punkte sind bereits im Master-CRS.
Lade Raster-Array: Hoehe...
Lade Raster-Array: SCA...
Lade Raster-Array: Slope...
Lade Raster-Array: TWI...
Lade Raster-Array: Year...
Alle Daten geladen.
Berechne Pixel-Koordinaten (r, c) für alle Punkte...
Pixel-Koordinaten erfolgreich berechnet.
--- Verarbeite Raster: Hoehe ---
Berechne Metrik 1: Punktwert...
  > Reprojiziere Raster 'Hoehe' auf Master-Grid...
Berechne Metrik 2: Median der gesamten Patch-Fläche (Zonal)...
Berechne Metrik 3: Median der 100m-Nachbarschaft (auf Patch)...


Lokale Mediane (Hoehe): 100%|██████████| 2331/2331 [00:00<00:00, 10860.53it/s]


--- Verarbeite Raster: SCA ---
Berechne Metrik 1: Punktwert...
  > Reprojiziere Raster 'SCA' auf Master-Grid...
Berechne Metrik 2: Median der gesamten Patch-Fläche (Zonal)...
Berechne Metrik 3: Median der 100m-Nachbarschaft (auf Patch)...


Lokale Mediane (SCA): 100%|██████████| 2331/2331 [00:00<00:00, 8845.28it/s] 


--- Verarbeite Raster: Slope ---
Berechne Metrik 1: Punktwert...
  > Reprojiziere Raster 'Slope' auf Master-Grid...
Berechne Metrik 2: Median der gesamten Patch-Fläche (Zonal)...
Berechne Metrik 3: Median der 100m-Nachbarschaft (auf Patch)...


...

--- Verarbeite Raster: TWI ---
Berechne Metrik 1: Punktwert...
  > Reprojiziere Raster 'TWI' auf Master-Grid...
Berechne Metrik 2: Median der gesamten Patch-Fläche (Zonal)...
Berechne Metrik 3: Median der 100m-Nachbarschaft (auf Patch)...


Lokale Mediane (TWI): 100%|██████████| 2331/2331 [00:00<00:00, 8867.88it/s]


--- Verarbeite Raster: Year ---
Berechne Metrik 1: Punktwert...
  > Raster-Grid stimmt bereits überein. Überspringe Reprojektion.
Berechne Metrik 2: Median der gesamten Patch-Fläche (Zonal)...
Berechne Metrik 3: Median der 100m-Nachbarschaft (auf Patch)...


Lokale Mediane (Year): 100%|██████████| 2331/2331 [00:00<00:00, 6657.22it/s]


--- Alle Berechnungen abgeschlossen ---
Speichere finale GeoPackage-Datei in: C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\fichtelforst.gpkg, Layer: punkte_westpixel_ha_stats
--- Skript erfolgreich beendet ---


# Filter by Roads

In [18]:
import os
import sys
import geopandas as gpd
import numpy as np
from shapely.geometry import LineString

# --- 1. Konfiguration ---

# --- Pfade für die Eingabe-Punkte ---
POINTS_PATH = r"C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\fichtelforst.gpkg"
INPUT_LAYER_NAME = "punkte_westpixel_ha_stats" 

# --- Pfade Wege ---
ROADS_PATH = r"C:/Users/ge48neq/T-EDGE/Fichtelgebirge/02_Data_bearb/OeSM/OeSM_road_25835.gpkg"
ROADS_LAYER_NAME = "OeSM_road_25835"

# --- Filter-Parameter ---
BUFFER_DISTANCE_METERS = 100
TRANSECT_LENGTH_METERS = 100
ROADS_TO_EXCLUDE = [
    'primary', 'service', 'track_grade2', 'track_grade3', 'track_grade4'
]

# --- Output-Namen ---
OUTPUT_POINTS_LAYER_NAME = f"noRoads_points_{int(BUFFER_DISTANCE_METERS)}m"
OUTPUT_TRANSECTS_LAYER_NAME = f"noRoads_trans_{int(BUFFER_DISTANCE_METERS)}m"
OUTPUT_BUFFER_PATH = r"C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\fichtelforst.gpkg"


# --- 2. Haupt-Skript ---

def filter_transects_near_roads():
    print(f"--- Starte Filterung nach Straßenabstand ({BUFFER_DISTANCE_METERS}m) ---")

    # --- Schritt 1: Punkte laden ---
    try:
        print(f"Lese Punkte: {POINTS_PATH} (Layer: {INPUT_LAYER_NAME})")
        gdf_points = gpd.read_file(POINTS_PATH, layer=INPUT_LAYER_NAME)
        print(f"  {len(gdf_points)} Punkte geladen.")
        points_crs = gdf_points.crs
    except Exception as e:
        print(f"FEHLER beim Lesen der Punkt-Datei/Layer: {e}")
        return False

    # --- Schritt 2: Straßen laden und filtern ---
    try:
        print(f"Lese Straßen: {ROADS_PATH} (Layer: {ROADS_LAYER_NAME})")
        gdf_roads = gpd.read_file(ROADS_PATH, layer=ROADS_LAYER_NAME)
        print(f"  {len(gdf_roads)} Straßen-Segmente geladen.")
        
        if gdf_roads.crs != points_crs:
            print(f"  WARNUNG: CRS-Mismatch. Projiziere Straßen nach {points_crs}...")
            gdf_roads = gdf_roads.to_crs(points_crs)
            
        gdf_roads_filtered = gdf_roads[gdf_roads['fclass'].isin(ROADS_TO_EXCLUDE)]
        
        if gdf_roads_filtered.empty:
            print("INFO: Keine der auszuschließenden Straßen-Typen gefunden. Keine Filterung nötig.")
            gdf_points.to_file(POINTS_PATH, layer=OUTPUT_POINTS_LAYER_NAME, driver="GPKG")
            # Generiere Linien
            generate_and_save_transects(gdf_points, POINTS_PATH, OUTPUT_TRANSECTS_LAYER_NAME)
            print("Original-Punkte und -Transekte unter neuen Layern gespeichert.")
            return True
            
        print(f"  {len(gdf_roads_filtered)} störende Straßen-Segmente gefunden.")
        
    except Exception as e:
        print(f"FEHLER beim Laden oder Filtern der Straßen: {e}")
        return False

    # --- Schritt 3: Puffer erstellen (Exclusion Zone) ---
    try:
        print(f"Erstelle {BUFFER_DISTANCE_METERS}m Puffer um störende Straßen...")
        roads_buffer = gdf_roads_filtered.buffer(BUFFER_DISTANCE_METERS)
        exclusion_zone = roads_buffer.union_all()
        exclusion_gdf = gpd.GeoDataFrame(geometry=[exclusion_zone], crs=points_crs)
    except Exception as e:
        print(f"FEHLER beim Puffern der Straßen: {e}")
        return False

    # --- Schritt 4: Transekte aus Punkten generieren ---
    print(f"Generiere {TRANSECT_LENGTH_METERS}m-Transekte (West) aus Startpunkten...")
    transect_geometries = []
    # Index der Punkte für die Zuordnung behalten
    original_index = gdf_points.index
    
    for point in gdf_points.geometry:
        start_x, start_y = point.x, point.y
        # Linie nach Westen
        end_x = start_x - TRANSECT_LENGTH_METERS
        end_y = start_y
        transect_geometries.append(LineString([(start_x, start_y), (end_x, end_y)]))

    gdf_transects = gpd.GeoDataFrame(geometry=transect_geometries, crs=points_crs)
    # Index, um die Linien mit den Punkten zu verknüpfen
    gdf_transects.index = original_index

    # --- Schritt 5: Transekte filtern ---
    try:
        print("Führe räumliche Filterung durch (Prüfe Linien auf 'intersects')...")
        
        # Finde alle Transekte, die die Pufferzone SCHNEIDEN
        intersecting_transects = gpd.sjoin(gdf_transects, exclusion_gdf, predicate='intersects')
        
        # Identifiziere die Indizes der ungeeigneten Transekte
        bad_indices = intersecting_transects.index
        
        # Erstelle eine Maske für brauchbare Indizes (die NICHT in der 'bad_indices'-Liste sind)
        good_mask = ~gdf_points.index.isin(bad_indices)
        
        # Wende die Maske auf die PUNKTE und LINIEN an
        filtered_points_gdf = gdf_points[good_mask].copy()
        filtered_transects_gdf = gdf_transects[good_mask].copy()
        
        print(f"  FILTERUNG ABGESCHLOSSEN: {len(filtered_points_gdf)} von {len(gdf_points)} Punkten/Transekten beibehalten.")
        
    except Exception as e:
        print(f"FEHLER beim räumlichen Filtern: {e}")
        return False

    # --- Schritt 6: Speichern ---
    try:
        # 1. Speichere die GEFILTERTEN PUNKTE
        print(f"Speichere gefilterte PUNKTE nach: {POINTS_PATH} (Layer: {OUTPUT_POINTS_LAYER_NAME})")
        filtered_points_gdf.to_file(POINTS_PATH, layer=OUTPUT_POINTS_LAYER_NAME, driver="GPKG")
        
        # 2. Speichere die GEFILTERTEN TRANSEKTE
        print(f"Speichere gefilterte TRANSEKTE nach: {POINTS_PATH} (Layer: {OUTPUT_TRANSECTS_LAYER_NAME})")
        filtered_transects_gdf.to_file(POINTS_PATH, layer=OUTPUT_TRANSECTS_LAYER_NAME, driver="GPKG")
        
        # 3. Speichere den Puffer
        print(f"Speichere Straßen-Puffer (Exclusion Zone) nach: {OUTPUT_BUFFER_PATH}")
        exclusion_gdf.to_file(OUTPUT_BUFFER_PATH, layer= f"roadsBuffer{BUFFER_DISTANCE_METERS}", driver="GPKG")
        
        
        print("  Erfolgreich gespeichert.")
        return True
    except Exception as e:
        print(f"FEHLER beim Speichern der Ergebnis-Dateien: {e}")
        return False

# --- Hilfsfunktion für den "if empty"-Fall ---
def generate_and_save_transects(points_gdf, output_path, layer_name):
    """Generiert Linien und speichert sie."""
    transect_geometries = []
    for point in points_gdf.geometry:
        start_x, start_y = point.x, point.y
        end_x = start_x - TRANSECT_LENGTH_METERS
        end_y = start_y
        transect_geometries.append(LineString([(start_x, start_y), (end_x, end_y)]))
    
    gdf_transects = gpd.GeoDataFrame(geometry=transect_geometries, crs=points_gdf.crs)
    gdf_transects.index = points_gdf.index
    gdf_transects.to_file(output_path, layer=layer_name, driver="GPKG")
    print(f"  Transekte gespeichert in: {output_path} (Layer: {layer_name})")


# --- 3. Skript ausführen ---
if __name__ == "__main__":
    success = filter_transects_near_roads()
    
    print("\n---------------------------------")
    if success:
        print("Skript erfolgreich beendet.")
    else:
        print("Skript mit Fehlern beendet.")

--- Starte Filterung nach Straßenabstand (100m) ---
Lese Punkte: C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\fichtelforst.gpkg (Layer: punkte_westpixel_ha_stats)
  2331 Punkte geladen.
Lese Straßen: C:/Users/ge48neq/T-EDGE/Fichtelgebirge/02_Data_bearb/OeSM/OeSM_road_25835.gpkg (Layer: OeSM_road_25835)
  114344 Straßen-Segmente geladen.
  46569 störende Straßen-Segmente gefunden.
Erstelle 100m Puffer um störende Straßen...
Generiere 100m-Transekte (West) aus Startpunkten...
Führe räumliche Filterung durch (Prüfe Linien auf 'intersects')...
  FILTERUNG ABGESCHLOSSEN: 661 von 2331 Punkten/Transekten beibehalten.
Speichere gefilterte PUNKTE nach: C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\fichtelforst.gpkg (Layer: noRoads_points_100m)
Speichere gefilterte TRANSEKTE nach: C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\fichtelforst.gpkg (Layer: noRoads_trans_100m)
Speichere Straßen-Puffer (Exclusion Zo

# Erstelle Samplepoints 

In [19]:
import os
import sys
import geopandas as gpd
from shapely.geometry import Point, MultiPoint

# --- 1. Konfiguration ---

# Pfad zur GeoPackage
POINTS_PATH = r"C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\fichtelforst.gpkg"
INPUT_LAYER_NAME = "noRoads_points_100m" 

# --- Definition des Mess-Designs ---

# Punkte entlang der Hauptlinie (in Metern vom West-Rand)
# -10 = 10m in die Störfläche (Osten)
# 0 = West-Rand
# 100 = 100m in den Wald (Westen)
TRANSECT_INTERVALS_M = [
    -10, 0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100
]
# 14 Intervalle

# Quer-Abstand (links/rechts bzw. Nord/Süd) in Metern
CROSS_OFFSET_M = 1.5

# --- Output ---
# Name des neuen Layers für die MultiPoint-Features
OUTPUT_LAYER_NAME = "final_survey_multipoints"

# --- 2. Haupt-Skript ---

def create_survey_multipoints():
    print(f"--- Erstelle QField-Survey-Punkte (MultiPoint-Layout) ---")

    # --- Schritt 1: Punkte laden ---
    try:
        print(f"Lese finale Start-Punkte: {POINTS_PATH} (Layer: {INPUT_LAYER_NAME})")
        gdf_points = gpd.read_file(POINTS_PATH, layer=INPUT_LAYER_NAME)
        print(f"  {len(gdf_points)} Transekt-Startpunkte geladen.")
        if gdf_points.empty:
            print("FEHLER: Keine Punkte im Layer gefunden. Breche ab.")
            return False
        
    except Exception as e:
        print(f"FEHLER beim Lesen der Punkt-Datei/Layer: {e}")
        return False

    # --- Schritt 2: MultiPoint-Geometrien erstellen ---
    print(f"Generiere MultiPoint-Layout für {len(gdf_points)} Transekte...")
    
    all_transect_multipoints = [] 

    # .itertuples() ist schnell und liest alle Attribute mit
    for row in gdf_points.itertuples(index=False): 
        
        start_point_geom = row.geometry
        start_x, start_y = start_point_geom.x, start_point_geom.y
        
        points_for_this_transect = [] 
        
        # Innere Schleife: Gehe alle Mess-Abstände durch
        for distance_m in TRANSECT_INTERVALS_M:
            
            # X-Koordinate: West ist negatives X.
            # -10m wird zu start_x + 10m
            # +100m wird zu start_x - 100m
            center_x = start_x - distance_m 
            
            # Punkt 1: Auf der Linie
            p_center = Point(center_x, start_y)
            
            # Punkt 2: 1.5m "links" (Nord)
            p_north = Point(center_x, start_y + CROSS_OFFSET_M)
            
            # Punkt 3: 1.5m "rechts" (Süd)
            p_south = Point(center_x, start_y - CROSS_OFFSET_M)
            
            points_for_this_transect.extend([p_center, p_north, p_south])
            
        # (Schleife fertig: 14 Intervalle * 3 Punkte = 42 Punkte sind gesammelt)
        
        # Erstelle die MultiPoint-Geometrie
        multipoint_geom = MultiPoint(points_for_this_transect)
        
    
        
        # 1. Wandle die Zeile (ein 'namedtuple') in ein Dictionary um.
        #    übernimmt automatisch alle Spalten (PatchID, Area_ha, etc.)
        new_data = row._asdict()
        
        # 2. Ersetze die ALTE Geometrie (der Start-Punkt) 
        #    durch die MultiPoint-Geometrie
        new_data['geometry'] = multipoint_geom
        
        # 3. Hänge das vollständige Dictionary an die Liste an
        all_transect_multipoints.append(new_data)


    # --- Schritt 3: Neues GeoDataFrame erstellen und speichern ---
    try:
        print(f"Erstelle finales GeoDataFrame mit {len(all_transect_multipoints)} MultiPoint-Features...")
        
        # Erstelle das GeoDataFrame mit dem CRS der Punkte
        gdf_final = gpd.GeoDataFrame(all_transect_multipoints, crs=gdf_points.crs)

        print(f"Speichere Survey-Punkte nach: {POINTS_PATH} (Layer: {OUTPUT_LAYER_NAME})")
        gdf_final.to_file(POINTS_PATH, layer=OUTPUT_LAYER_NAME, driver="GPKG")
        
        print("  Erfolgreich gespeichert.")
        return True
    except Exception as e:
        print(f"FEHLER beim Erstellen oder Speichern des GeoDataFrames: {e}")
        return False

# --- 3. Skript ausführen ---
if __name__ == "__main__":
    success = create_survey_multipoints()
    
    print("\n---------------------------------")
    if success:
        print("Skript erfolgreich beendet.")
    else:
        print("Skript mit Fehlern beendet.")

--- Erstelle QField-Survey-Punkte (MultiPoint-Layout) ---
Lese finale Start-Punkte: C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\fichtelforst.gpkg (Layer: noRoads_points_100m)
  661 Transekt-Startpunkte geladen.
Generiere MultiPoint-Layout für 661 Transekte...
Erstelle finales GeoDataFrame mit 661 MultiPoint-Features...
Speichere Survey-Punkte nach: C:\Users\ge48neq\T-EDGE\Fichtelgebirge\02_Data_bearb\Alba_dist_combined_v2\fichtelforst.gpkg (Layer: final_survey_multipoints)
  Erfolgreich gespeichert.

---------------------------------
Skript erfolgreich beendet.
